# Phase 6: TS1 Mechanism (~121 advected species, ~592 reactions)

Verifies the full TS1 tropospheric/stratospheric mechanism:
- Config-only switch from Chapman (no recompilation needed)
- 121 advected species, 209 total (including short-lived radicals)
- 592 reactions: 361 Arrhenius, 142 user-defined, 31 Troe, 24 emission, 21 first-order loss, 13 surface
- TUV-x photolysis (72 of 73 mapped), 69 static rate parameters
- O3, NO, NO2, CO show expected diurnal patterns
- No negative species concentrations

**Pre-requisite:** `data/jw_480km_ts1/output.nc`

In [ ]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

DATA_DIR = Path("..") / "data" / "jw_480km_ts1"
OUTPUT = DATA_DIR / "output.nc"
LOG = DATA_DIR / "log.atmosphere.0000.out"

assert OUTPUT.exists(), f"Run the TS1 test first — {OUTPUT} not found"

ds = nc.Dataset(OUTPUT)
lat = np.degrees(ds["latCell"][:])
lon = np.degrees(ds["lonCell"][:])
area = ds["areaCell"][:]
nCells = ds.dimensions["nCells"].size
nTimes = ds.dimensions["Time"].size
nLevels = ds.dimensions["nVertLevels"].size
print(f"Grid: {nCells} cells, {nLevels} levels, {nTimes} time steps")

# Identify chemistry species (3D vars that aren't dynamics/tracers)
DYNAMICS = {"pressure_base", "pressure_p", "theta",
            "uReconstructZonal", "uReconstructMeridional",
            "qv", "tracer_1", "tracer_2", "tracer_3"}
chem_vars = sorted([v for v in ds.variables
                    if len(ds[v].dimensions) == 3 and v not in DYNAMICS])
print(f"Chemistry species in output: {len(chem_vars)}")

## 1. Log Verification

Parse the MPAS log file to verify chemistry initialization: species counts,
emissions/deposition, TUV-x photolysis mapping, and static rate parameters.

In [ ]:
log_text = LOG.read_text()

checks = {
    "Scalars extended": r"Extended scalars: 4 → (\d+) constituents",
    "MICM species": r"MICM: (\d+) species, (\d+) rate params",
    "Advected species": r"Species: (\d+) advected",
    "Emissions": r"Emissions: (\d+) species",
    "Deposition": r"Deposition: (\d+) species",
    "TUV-x reactions": r"TUV-x: (\d+) photolysis reactions",
    "TUV-x mapping": r"photo mapping: (\d+) of (\d+)",
    "Static rates": r"Static rate params: (\d+) loaded",
    "Init complete": r"Chemistry initialization complete",
}

all_pass = True
for label, pattern in checks.items():
    m = re.search(pattern, log_text)
    if m:
        print(f"  PASS  {label}: {m.group(0).split('] ')[-1] if '] ' in m.group(0) else m.group(0)}")
    else:
        print(f"  FAIL  {label}: pattern not found")
        all_pass = False

assert all_pass, "Some log checks failed"
print("\nAll log checks passed.")

## 2. Key Species Diurnal Cycles

Domain-mean mixing ratios over time for O3, NO, NO2, CO at a
representative level (~500 hPa, level 6). These species should show
temporal evolution driven by photolysis, emissions, and deposition.

In [ ]:
key_species = ["o3", "no", "no2", "co"]
lev = 6  # ~500 hPa
hours = np.arange(nTimes)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, sp in zip(axes.flat, key_species):
    data = ds[sp][:, :, lev]  # (Time, nCells)
    mean = data.mean(axis=1)
    vmin = data.min(axis=1)
    vmax = data.max(axis=1)
    ax.fill_between(hours, vmin * 1e6, vmax * 1e6, alpha=0.2)
    ax.plot(hours, mean * 1e6, "o-", markersize=3)
    ax.set_title(sp.upper())
    ax.set_xlabel("Time step (hours)")
    ax.set_ylabel("ppmv")
    ax.grid(True, alpha=0.3)

plt.suptitle(f"Key Species Diurnal Cycles at level {lev}", fontsize=14)
plt.tight_layout()
plt.show()

# Verify O3 is physically reasonable
o3_all = ds["o3"][:]
print(f"O3 range: [{o3_all.min():.3e}, {o3_all.max():.3e}] kg/kg")
assert o3_all.max() < 1e-3, "O3 unreasonably large"
print("O3 values physically reasonable — PASS")

## 3. O3 Spatial Distribution

Map of O3 at ~500 hPa for the initial and final time steps, showing
any dayside/nightside contrast from photolysis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, t, label in zip(axes, [0, -1], ["Initial", "Final (24 h)"]):
    vals = ds["o3"][t, :, lev] * 1e6
    sc = ax.scatter(lon, lat, c=vals, s=4, cmap="YlGn")
    ax.set_xlabel("Longitude (deg)")
    ax.set_ylabel("Latitude (deg)")
    ax.set_title(f"O3 at level {lev} — {label}")
    plt.colorbar(sc, ax=ax, label="ppmv")
plt.tight_layout()
plt.show()

## 4. Emission Species Accumulation

With stub emissions (constant surface flux) and deposition for some species,
we expect emitted-only species (e.g., CO) to accumulate at the surface, while
species with both emission and deposition (e.g., SO2) should approach equilibrium.

In [ ]:
# Species with emission only (no deposition)
emit_only = ["co", "isop", "c2h6", "c3h8", "bigalk", "toluene",
             "xylenes", "benzene", "mek", "hcn", "c2h5oh",
             "c2h4", "c3h6", "bigene", "dms"]
# Species with both emission and deposition
emit_dep = ["so2", "ch2o", "hcooh", "ch3cooh", "nh3", "ch3cho"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Emission-only: surface mean should increase
ax = axes[0]
for sp in emit_only[:6]:  # Plot a selection
    sfc = np.array([ds[sp][t, :, -1].mean() for t in range(nTimes)])
    ax.plot(hours, sfc * 1e9, "o-", markersize=2, label=sp.upper())
ax.set_xlabel("Time step (hours)")
ax.set_ylabel("ppbv")
ax.set_title("Emission-only species (surface)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Emission + deposition: should show deceleration
ax = axes[1]
for sp in emit_dep:
    sfc = np.array([ds[sp][t, :, -1].mean() for t in range(nTimes)])
    ax.plot(hours, sfc * 1e9, "o-", markersize=2, label=sp.upper())
ax.set_xlabel("Time step (hours)")
ax.set_ylabel("ppbv")
ax.set_title("Emission + deposition species (surface)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Verify CO accumulation at surface
co_sfc = np.array([ds["co"][t, :, -1].mean() for t in range(nTimes)])
co_diff = np.diff(co_sfc)
assert co_sfc[-1] > co_sfc[0], "CO should accumulate from emissions"
print(f"CO surface: {co_sfc[0]*1e9:.2f} -> {co_sfc[-1]*1e9:.2f} ppbv — accumulation PASS")

## 5. Vertical Profile Structure

Verify that species have a reasonable vertical structure at the final time step.
O3 typically increases with altitude; NO concentrations are higher at the surface
(from emissions) and in the upper troposphere.

In [ ]:
profile_species = ["o3", "no", "no2", "co", "ch2o", "hno3"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, sp in zip(axes.flat, profile_species):
    # Global mean profile at final time step
    profile = ds[sp][-1, :, :].mean(axis=0)
    ax.plot(profile * 1e9, range(nLevels), "b-", linewidth=2)
    ax.set_title(sp.upper())
    ax.set_xlabel("ppbv")
    ax.set_ylabel("Level (0=top)")
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3)

plt.suptitle("Global-mean Vertical Profiles (final time step)", fontsize=14)
plt.tight_layout()
plt.show()

## 6. Negative Species Check

All advected species should remain non-negative after 24 hours of simulation.

In [ ]:
print(f"Checking {len(chem_vars)} species for negative values...")
negatives = []
for v in chem_vars:
    min_val = ds[v][:].min()
    if min_val < 0:
        negatives.append((v, float(min_val)))

if negatives:
    print(f"\n  WARNING: {len(negatives)} species have negative values:")
    for v, val in sorted(negatives, key=lambda x: x[1]):
        print(f"    {v:20s}: min = {val:.3e}")
    print("\n  (Small negatives from advection are acceptable)")
else:
    print("  All species non-negative — PASS")

## 7. Summary

Collect all results and report overall pass/fail status.

In [ ]:
print("=" * 60)
print("Phase 6: TS1 Mechanism Verification Summary")
print("=" * 60)

results = []

# 1. Species count
n_expected = 121
results.append(("Species in output", len(chem_vars) == n_expected,
                f"{len(chem_vars)}/{n_expected}"))

# 2. O3 physical range
o3_ok = o3_all.max() < 1e-3 and o3_all.min() >= -1e-10
results.append(("O3 physical range", o3_ok,
                f"[{o3_all.min():.2e}, {o3_all.max():.2e}]"))

# 3. CO accumulation from emissions
co_accum = co_sfc[-1] > co_sfc[0]
results.append(("CO surface accumulation", co_accum,
                f"{co_sfc[0]*1e9:.2f} -> {co_sfc[-1]*1e9:.2f} ppbv"))

# 4. Negative species
n_neg = len(negatives)
severe_neg = [v for v, val in negatives if val < -1e-10]
results.append(("No severe negatives", len(severe_neg) == 0,
                f"{n_neg} species with negatives, {len(severe_neg)} severe"))

# 5. Log checks
results.append(("Log initialization checks", all_pass, "All patterns found"))

for label, passed, detail in results:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}]  {label}: {detail}")

print("=" * 60)
all_ok = all(p for _, p, _ in results)
if all_ok:
    print("Phase 6 PASSED — TS1 mechanism working correctly")
else:
    print("Phase 6 FAILED — see above for details")

ds.close()